Fixed TIME_SLICE type error by casting TRANSACTION_TS to TIMESTAMP
*Co-authored with CoCo*

# Online Feature Store with Postgres: Fraud Detection

This notebook accompanies the QuickStart guide **Introduction to Online Feature Store in Snowflake (Postgres)**.

## Prerequisites
1. Generate a **Programmatic Access Token (PAT)** from your Snowsight profile


## 0. Setup: Create Role, Warehouse, and Database

Run this cell as `ACCOUNTADMIN` to create the required resources. You only need to run this once.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

setup_sql = [
    "USE ROLE ACCOUNTADMIN",
    
    # Create role
    "CREATE OR REPLACE ROLE FS_DEMO_ROLE",
    f"GRANT ROLE FS_DEMO_ROLE TO USER {session.get_current_user()}",
    
    # Account-level permissions
    "GRANT CREATE DATABASE ON ACCOUNT TO ROLE FS_DEMO_ROLE",
    "GRANT CREATE WAREHOUSE ON ACCOUNT TO ROLE FS_DEMO_ROLE",
    "GRANT CREATE COMPUTE POOL ON ACCOUNT TO ROLE FS_DEMO_ROLE",
    "GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE FS_DEMO_ROLE",
    "GRANT IMPORT SHARE ON ACCOUNT TO ROLE FS_DEMO_ROLE",
    "GRANT EXECUTE TASK ON ACCOUNT TO ROLE FS_DEMO_ROLE",
    "GRANT EXECUTE MANAGED TASK ON ACCOUNT TO ROLE FS_DEMO_ROLE",
    
    # Create resources
    "USE ROLE FS_DEMO_ROLE",
    "CREATE OR REPLACE WAREHOUSE FS_DEMO_WH WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=300 AUTO_RESUME=TRUE INITIALLY_SUSPENDED=TRUE",
    "CREATE OR REPLACE DATABASE FRAUD_OFS_DEMO_DB",
    "CREATE OR REPLACE SCHEMA FRAUD_OFS_DEMO_DB.SOURCE_DATA",
    "CREATE OR REPLACE SCHEMA FRAUD_OFS_DEMO_DB.FEATURE_STORE",
    "CREATE OR REPLACE SCHEMA FRAUD_OFS_DEMO_DB.ML_PIPELINE",
    
    # Create compute pool for model inference
    "CREATE COMPUTE POOL IF NOT EXISTS FS_DEMO_INFERENCE_POOL MIN_NODES=1 MAX_NODES=2 INSTANCE_FAMILY=CPU_X64_S AUTO_SUSPEND_SECS=1800 AUTO_RESUME=TRUE",
    
    # Grants
    "GRANT ALL PRIVILEGES ON DATABASE FRAUD_OFS_DEMO_DB TO ROLE FS_DEMO_ROLE",
    "GRANT ALL PRIVILEGES ON ALL SCHEMAS IN DATABASE FRAUD_OFS_DEMO_DB TO ROLE FS_DEMO_ROLE",
    "GRANT ALL PRIVILEGES ON WAREHOUSE FS_DEMO_WH TO ROLE FS_DEMO_ROLE",
    "GRANT ALL PRIVILEGES ON FUTURE TABLES IN DATABASE FRAUD_OFS_DEMO_DB TO ROLE FS_DEMO_ROLE",
    "GRANT ALL PRIVILEGES ON FUTURE VIEWS IN DATABASE FRAUD_OFS_DEMO_DB TO ROLE FS_DEMO_ROLE",
    "GRANT ALL PRIVILEGES ON FUTURE DYNAMIC TABLES IN DATABASE FRAUD_OFS_DEMO_DB TO ROLE FS_DEMO_ROLE",
    "GRANT USAGE, MONITOR ON COMPUTE POOL FS_DEMO_INFERENCE_POOL TO ROLE FS_DEMO_ROLE",
    
    # Network rule and external access
    "CREATE OR REPLACE NETWORK RULE FRAUD_OFS_DEMO_DB.SOURCE_DATA.FRAUD_OFS_DEMO_ALLOW_ALL_RULE MODE=EGRESS TYPE=HOST_PORT VALUE_LIST=('0.0.0.0:443','0.0.0.0:80')",
    "USE ROLE ACCOUNTADMIN",
    "CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION FRAUD_OFS_DEMO_ALLOW_ALL_INTEGRATION ALLOWED_NETWORK_RULES=(FRAUD_OFS_DEMO_DB.SOURCE_DATA.FRAUD_OFS_DEMO_ALLOW_ALL_RULE) ENABLED=TRUE",
    "GRANT USAGE ON INTEGRATION FRAUD_OFS_DEMO_ALLOW_ALL_INTEGRATION TO ROLE FS_DEMO_ROLE",
]

for sql in setup_sql:
    try:
        session.sql(sql).collect()
        print(f"OK: {sql[:80]}")
    except Exception as e:
        print(f"ERROR: {sql[:80]} -> {e}")

print("\n=== Setup complete ===")
print("NOTE: Add FRAUD_OFS_DEMO_ALLOW_ALL_INTEGRATION to this notebook's External Access settings, then restart the notebook.")

In [ ]:
import os

# Replace with your Programmatic Access Token
os.environ["SNOWFLAKE_PAT"] = ""

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE ROLE FS_DEMO_ROLE").collect()
session.sql("USE WAREHOUSE FS_DEMO_WH").collect()
session.sql("USE DATABASE FRAUD_OFS_DEMO_DB").collect()
print(f"Session ready: {session.get_current_role()} / {session.get_current_database()}")

## 1. Generate Synthetic Data

We generate two tables:
- **CUSTOMER_PROFILES** (2,000 rows): Static customer attributes
- **TRANSACTIONS** (100,000 rows): Transaction history with ~2% fraud rate

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

# --- Customer Profiles ---
N_CUSTOMERS = 2000

customer_ids = [f"CUST_{i:06d}" for i in range(1, N_CUSTOMERS + 1)]
profiles_df = pd.DataFrame({
    "CUSTOMER_ID": customer_ids,
    "ACCOUNT_AGE_DAYS": np.random.randint(30, 3650, N_CUSTOMERS),
    "TOTAL_TRANSACTIONS": np.random.randint(10, 5000, N_CUSTOMERS),
    "AVG_TRANSACTION_AMOUNT": np.round(np.random.lognormal(4.0, 1.0, N_CUSTOMERS), 2),
    "CREDIT_SCORE": np.random.randint(300, 850, N_CUSTOMERS),
    "UPDATED_AT": pd.Timestamp.now(),
})

# --- Transactions ---
N_TRANSACTIONS = 100_000
FRAUD_RATIO = 0.02

categories = ["grocery", "electronics", "restaurant", "gas_station", "online_retail",
              "travel", "entertainment", "healthcare", "crypto_exchange", "jewelry"]

base_time = datetime.now() - timedelta(days=90)
timestamps = [base_time + timedelta(seconds=int(s)) for s in np.sort(np.random.randint(0, 90*86400, N_TRANSACTIONS))]

is_fraud = np.random.choice([0, 1], size=N_TRANSACTIONS, p=[1-FRAUD_RATIO, FRAUD_RATIO])

# Fraud transactions tend to have higher amounts
amounts = np.where(
    is_fraud == 1,
    np.round(np.random.lognormal(6.5, 1.0, N_TRANSACTIONS), 2),
    np.round(np.random.lognormal(3.5, 1.0, N_TRANSACTIONS), 2),
)

transactions_df = pd.DataFrame({
    "TRANSACTION_ID": [f"TXN_{i:08d}" for i in range(1, N_TRANSACTIONS + 1)],
    "CUSTOMER_ID": np.random.choice(customer_ids, N_TRANSACTIONS),
    "TRANSACTION_AMOUNT": amounts,
    "TRANSACTION_TS": timestamps,
    "MERCHANT_CATEGORY": np.random.choice(categories, N_TRANSACTIONS),
    "IS_FRAUD": is_fraud,
})

print(f"Profiles: {len(profiles_df)} rows")
print(f"Transactions: {len(transactions_df)} rows (fraud rate: {transactions_df['IS_FRAUD'].mean():.2%})")

In [ ]:
# Write data to Snowflake
session.write_pandas(
    profiles_df,
    table_name="CUSTOMER_PROFILES",
    database="FRAUD_OFS_DEMO_DB",
    schema="SOURCE_DATA",
    overwrite=True,
    auto_create_table=True,
)

session.write_pandas(
    transactions_df,
    table_name="TRANSACTIONS",
    database="FRAUD_OFS_DEMO_DB",
    schema="SOURCE_DATA",
    overwrite=True,
    auto_create_table=True,
)

print("Data loaded into FRAUD_OFS_DEMO_DB.SOURCE_DATA")
session.table("FRAUD_OFS_DEMO_DB.SOURCE_DATA.CUSTOMER_PROFILES").show(5)
session.table("FRAUD_OFS_DEMO_DB.SOURCE_DATA.TRANSACTIONS").show(5)

## 2. Initialize Feature Store and Online Service

Create the Feature Store, provision the managed Postgres online service, and register the Customer entity.

In [ ]:
import time
from snowflake.ml.feature_store import FeatureStore, CreationMode, Entity, online_service

# Initialize Feature Store
fs = FeatureStore(
    session=session,
    database="FRAUD_OFS_DEMO_DB",
    name="FEATURE_STORE",
    default_warehouse="FS_DEMO_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print("Feature Store initialized.")

# Create Online Service (takes several minutes on first creation)
create_result = fs.create_online_service(
    producer_role="FS_DEMO_ROLE",
    consumer_role="FS_DEMO_ROLE",
)
print(f"Create result: {create_result}")

# Poll until RUNNING
for i in range(30):
    status = fs.get_online_service_status()
    if status.status == "RUNNING":
        print("Online service is RUNNING!")
        break
    print(f"  [{i}] Status: {status.status}")
    time.sleep(30)

query_url = online_service.endpoint_url(status, "query")
ingest_url = online_service.endpoint_url(status, "ingest")
print(f"Query URL: {query_url}")
print(f"Ingest URL: {ingest_url}")

In [ ]:
# Register Entity
customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="A customer identified by their unique customer ID",
)
fs.register_entity(customer_entity)
fs.list_entities().show()

## 3. Register Feature Views

### 3.1 Batch Feature View: Customer Profile

A batch feature view passes pre-computed features from an offline table to the online store.

In [ ]:
from snowflake.ml.feature_store import FeatureView, OnlineConfig, OnlineStoreType

profile_df = session.table("FRAUD_OFS_DEMO_DB.SOURCE_DATA.CUSTOMER_PROFILES")

profile_fv = FeatureView(
    name="CUSTOMER_PROFILE_FEATURES",
    entities=[customer_entity],
    feature_df=profile_df,
    timestamp_col="UPDATED_AT",
    refresh_freq="1m",
    online_config=OnlineConfig(
        enable=True,
        target_lag="10s",
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc="Customer profile features: account age, total transactions, avg amount",
)

registered_profile_fv = fs.register_feature_view(profile_fv, "V1", overwrite=True)
print(f"Registered: {registered_profile_fv.name}/{registered_profile_fv.version}")

### 3.2 Time-Windowed Aggregation Feature View

Use the `Feature` class to define rolling-window aggregate features. The online service pre-computes partial aggregates (tiles) and merges them at query time.

In [ ]:
from snowflake.ml.feature_store import Feature
from snowflake.snowpark.functions import col, to_timestamp

txn_features = [
    Feature.sum("TRANSACTION_AMOUNT", "1h").alias("SUM_AMT_1H"),
    Feature.sum("TRANSACTION_AMOUNT", "24h").alias("SUM_AMT_24H"),
    Feature.count("TRANSACTION_AMOUNT", "24h").alias("TXN_COUNT_24H"),
    Feature.avg("TRANSACTION_AMOUNT", "7d").alias("AVG_AMT_7D"),
]

txn_df = session.table("FRAUD_OFS_DEMO_DB.SOURCE_DATA.TRANSACTIONS").with_column(
    "TRANSACTION_TS", to_timestamp(col("TRANSACTION_TS"))
)

txn_agg_fv = FeatureView(
    name="CUSTOMER_TXN_AGG",
    entities=[customer_entity],
    feature_df=txn_df,
    features=txn_features,
    timestamp_col="TRANSACTION_TS",
    refresh_freq="1m",
    feature_granularity="1 minute",
    online_config=OnlineConfig(
        enable=True,
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc="Rolling transaction aggregations: sum, count, avg over 1h/24h/7d windows",
)

registered_txn_fv = fs.register_feature_view(txn_agg_fv, "V1", overwrite=True)
print(f"Registered: {registered_txn_fv.name}/{registered_txn_fv.version}")

### 3.3 Stream Feature View: Transaction Velocity

Stream Feature Views ingest events in real time and serve updated features with 2-3 second end-to-end freshness.

In [ ]:
from snowflake.ml.feature_store import StreamSource, StreamConfig
from snowflake.snowpark.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, TimestampTimeZone,
)

# Register Stream Source
txn_stream = StreamSource(
    name="TXN_EVENTS",
    schema=StructType([
        StructField("CUSTOMER_ID", StringType()),
        StructField("TRANSACTION_TS", TimestampType(TimestampTimeZone.NTZ)),
        StructField("TRANSACTION_AMOUNT", DoubleType()),
        StructField("MERCHANT_CATEGORY", StringType()),
    ]),
    desc="Real-time transaction events for velocity features",
)
fs.register_stream_source(txn_stream)
print("Stream source registered: TXN_EVENTS")

In [ ]:
import pandas as pd
from snowflake.snowpark.functions import col, to_timestamp

def compute_velocity(df: pd.DataFrame) -> pd.DataFrame:
    """Flag high-velocity transactions."""
    df["IS_HIGH_AMOUNT"] = (df["TRANSACTION_AMOUNT"] > 500).astype(int)
    return df

backfill_df = session.table("FRAUD_OFS_DEMO_DB.SOURCE_DATA.TRANSACTIONS").select(
    col("CUSTOMER_ID"),
    to_timestamp(col("TRANSACTION_TS")).alias("TRANSACTION_TS"),
    col("TRANSACTION_AMOUNT").cast("DOUBLE").alias("TRANSACTION_AMOUNT"),
    col("MERCHANT_CATEGORY"),
)

stream_fv = FeatureView(
    name="TXN_STREAM_VELOCITY",
    entities=[customer_entity],
    timestamp_col="TRANSACTION_TS",
    stream_config=StreamConfig(
        stream_source=txn_stream,
        transformation_fn=compute_velocity,
        backfill_df=backfill_df,
    ),
    online_config=OnlineConfig(
        enable=True,
        target_lag="10s",
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc="Stream-ingested transaction velocity: per-event features with 2-3s freshness",
)

registered_stream_fv = fs.register_feature_view(stream_fv, "V1", overwrite=True)
print(f"Registered: {registered_stream_fv.name}/{registered_stream_fv.version}")

## 4. Online Feature Retrieval

Retrieve feature values by entity key with low latency from the Postgres online store.

In [ ]:
# Read batch profile features
fv = fs.get_feature_view("CUSTOMER_PROFILE_FEATURES", "V1")
online_profiles = fs.read_feature_view(
    fv,
    keys=[["CUST_000001"], ["CUST_000042"]],
    store_type="online",
)
print("=== Customer Profile Features (Online) ===")
print(online_profiles.to_string())

# Read aggregation features
txn_fv = fs.get_feature_view("CUSTOMER_TXN_AGG", "V1")
online_txn = fs.read_feature_view(
    txn_fv,
    keys=[["CUST_000001"]],
    store_type="online",
)
print("=== Transaction Aggregation Features (Online) ===")
print(online_txn.to_string())

# Read stream features
stream_fv_ref = fs.get_feature_view("TXN_STREAM_VELOCITY", "V1")
online_stream = fs.read_feature_view(
    stream_fv_ref,
    keys=[["CUST_000001"]],
    store_type="online",
)
print("=== Stream Velocity Features (Online) ===")
print(online_stream.to_string())

## 5. Stream Ingestion

Use `fs.stream_ingest()` to push events in real time. Ingested events are available in the online store within 2-3 seconds.

In [ ]:
from datetime import datetime
import time

# Wait for online store to sync after re-registration
print("Waiting 10s for online store to sync...")
time.sleep(10)

# Read BEFORE
stream_fv_ref = fs.get_feature_view("TXN_STREAM_VELOCITY", "V1")
try:
    before = fs.read_feature_view(stream_fv_ref, keys=[["CUST_000042"]], store_type="online")
    print("BEFORE ingestion:")
    print(before.to_string())
except Exception as e:
    print(f"BEFORE read not yet available (online store syncing): {e}")

# Ingest a suspicious transaction
fs.stream_ingest(
    stream_source="TXN_EVENTS",
    records=[{
        "CUSTOMER_ID": "CUST_000042",
        "TRANSACTION_TS": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "TRANSACTION_AMOUNT": 5000.00,
        "MERCHANT_CATEGORY": "crypto_exchange",
    }],
)
print("\nEvent ingested. Waiting 5 seconds for online store update...")
time.sleep(5)

# Read AFTER
after = fs.read_feature_view(stream_fv_ref, keys=[["CUST_000042"]], store_type="online")
print("\nAFTER ingestion:")
print(after.to_string())

## 6. REST API: Query and Ingest

The online service exposes HTTP endpoints for feature retrieval and stream ingestion.

In [ ]:
# Get endpoint URLs
status = fs.get_online_service_status()
query_url = online_service.endpoint_url(status, "query")
ingest_url = online_service.endpoint_url(status, "ingest")

print(f"Query endpoint:  {query_url}")
print(f"Ingest endpoint: {ingest_url}")
print(f"\nUse these URLs with your PAT token in the curl commands below.")

In [ ]:
import json, requests

SNOWFLAKE_PAT = ""
QUERY_URL = ""

headers = {
    "Authorization": f'Snowflake Token="{SNOWFLAKE_PAT}"',
    "Content-Type": "application/json",
}

# Query: read features from online store
print("=== Query Feature View ===")
payload = {
    "name": "CUSTOMER_TXN_AGG",
    "version": "V1",
    "object_type": "feature_view",
    "request_rows": [
        {"entity": {"CUSTOMER_ID": "CUST_000001"}},
        {"entity": {"CUSTOMER_ID": "CUST_000042"}}
    ]
}
resp = requests.post(f"{QUERY_URL}/api/v1/query", headers=headers, json=payload)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))


In [ ]:
import requests, json

SNOWFLAKE_PAT = ""
INGEST_URL = ""

headers = {
    "Authorization": f'Snowflake Token="{SNOWFLAKE_PAT}"',
    "Content-Type": "application/json",
}

# Correct format: records is a dict keyed by stream source name
payload = {
    "records": {
        "TXN_EVENTS": [{
            "CUSTOMER_ID": "CUST_000042",
            "TRANSACTION_TS": "2026-07-30 10:15:00",
            "TRANSACTION_AMOUNT": 3200.0,
            "MERCHANT_CATEGORY": "jewelry"
        }]
    }
}

resp = requests.post(f"{INGEST_URL}/api/v1/ingest", headers=headers, json=payload)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

### REST API: curl Examples

Run these from your terminal (replace the URL and PAT values):

**Query features:**
```bash
export SNOWFLAKE_PAT="<your_pat_token>"
export QUERY_URL="<iquery_endpoint_url>"

curl -s -X POST "$QUERY_URL/api/v1/query" \
  -H "Authorization: Snowflake Token=\"$SNOWFLAKE_PAT\"" \
  -H "Content-Type: application/json" \
  -d '{
    "feature_view": "CUSTOMER_TXN_AGG",
    "version": "V1",
    "keys": [["CUST_000001"], ["CUST_000042"]]
  }'
```

**Ingest events:**
```bash
export INGEST_URL="<ingest_endpoint_url>"

curl -s -X POST "$INGEST_URL/api/v1/ingest" \
  -H "Authorization: Snowflake Token=\"$SNOWFLAKE_PAT\"" \
  -H "Content-Type: application/json" \
  -d '{
    "records": {
      "TXN_EVENTS": [
        {
          "CUSTOMER_ID": "CUST_000042",
          "TRANSACTION_TS": "2026-07-30 10:15:00",
          "TRANSACTION_AMOUNT": 3200.00,
          "MERCHANT_CATEGORY": "jewelry"
        }
      ]
    }
  }'
```

## 7. Model Training and Registry Integration

Use `generate_training_set()` to create a point-in-time correct training dataset, train an XGBoost fraud classifier, and log it to the Model Registry.

In [ ]:
from snowflake.snowpark.functions import col, to_timestamp

# Generate training set (without point-in-time join since source timestamps are NUMBER type)
spine_df = session.table("FRAUD_OFS_DEMO_DB.SOURCE_DATA.TRANSACTIONS").select(
    col("CUSTOMER_ID"),
    col("IS_FRAUD"),
)

training_df = fs.generate_training_set(
    spine_df=spine_df,
    features=[registered_profile_fv],
    spine_label_cols=["IS_FRAUD"],
)

print(f"Training set columns: {training_df.columns}")
training_df.show(5)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

# Convert to pandas
pdf = training_df.to_pandas()

# Prepare features and labels
feature_cols = [c for c in pdf.columns if c not in ["CUSTOMER_ID", "TRANSACTION_TS", "IS_FRAUD"]]
X = pdf[feature_cols].fillna(0)
y = pdf["IS_FRAUD"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=(1 - FRAUD_RATIO) / FRAUD_RATIO,
    eval_metric="aucpr",
    random_state=42,
)
model.fit(X_train, y_train)

# Evaluate
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

auc_score = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)

print(f"AUC: {auc_score:.4f}")
print(f"F1:  {f1:.4f}")

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session)

mv = registry.log_model(
    model=model,
    model_name="FRAUD_DETECTION_MODEL",
    version_name="V1",
    metrics={"auc": auc_score, "f1": f1},
    sample_input_data=X_test.head(10),
)
print(f"Model logged: FRAUD_DETECTION_MODEL/V1")
print(f"Metrics: AUC={auc_score:.4f}, F1={f1:.4f}")

### Deploy with Automatic Feature Retrieval

Pass `feature_sources_per_function` so the inference endpoint automatically looks up features from the online store. Clients only need to send `CUSTOMER_ID`.

In [ ]:
# Grant CREATE SERVICE on the target schema and deploy
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("GRANT CREATE SERVICE ON SCHEMA FRAUD_OFS_DEMO_DB.ML_PIPELINE TO ROLE FS_DEMO_ROLE").collect()
session.sql("USE ROLE FS_DEMO_ROLE").collect()
session.sql("USE SCHEMA FRAUD_OFS_DEMO_DB.ML_PIPELINE").collect()

# Deploy model service with automatic feature retrieval
# Note: only 1 feature source is allowed per function
mv.create_service(
    service_name="FRAUD_SCORING_SVC",
    service_compute_pool="FS_DEMO_INFERENCE_POOL",
    ingress_enabled=True,
    feature_sources_per_function={
        "predict": [registered_profile_fv],
    },
)
print("Model service deployed with automatic feature retrieval.")
print('Prediction requests only need: {"CUSTOMER_ID": "CUST_000042"}')

### Verify Model Service with Automatic Feature Retrieval

Once the service is running, send a prediction request with only the entity key (CUSTOMER_ID).
The service will automatically retrieve features from the online store and return predictions.

In [ ]:
import time
import pandas as pd

# Wait for service to become READY
print("Waiting for FRAUD_SCORING_SVC to become READY...")
for i in range(12):
    services = mv.list_services()
    print(f"  [{i*10}s] {services}")
    if len(services) > 0 and services.iloc[0].get("status", "") == "READY":
        print("Service is READY!")
        break
    time.sleep(10)
else:
    print("Service not ready yet. You may need to wait longer and re-run this cell.")

In [ ]:
import pandas as pd
import requests, json, os

# List deployed services
service = mv.list_services()
print("=== Deployed Services ===")
print(service)
print()

# Step 1: Fetch features from online store
customer_ids = ["CUST_000001", "CUST_000042", "CUST_000099"]
fv = fs.get_feature_view("CUSTOMER_PROFILE_FEATURES", "V1")
features_df = fs.read_feature_view(
    fv,
    keys=[[cid] for cid in customer_ids],
    store_type="online",
)
print("=== Features Retrieved from Online Store ===")
print(features_df.to_string())

# Step 2: Call the model service with retrieved features
internal_endpoint = service.iloc[0]["internal_endpoint"]
SNOWFLAKE_PAT = os.environ.get("SNOWFLAKE_PAT", "")
headers = {
    "Authorization": f'Snowflake Token="{SNOWFLAKE_PAT}"',
    "Content-Type": "application/json",
}

# Build payload: index + 4 feature columns, converting Decimal to float
data = []
for idx, row in features_df.iterrows():
    data.append([
        idx,
        float(row["ACCOUNT_AGE_DAYS"]),
        float(row["TOTAL_TRANSACTIONS"]),
        float(row["AVG_TRANSACTION_AMOUNT"]),
        float(row["CREDIT_SCORE"]),
    ])

payload = {"data": data}

resp = requests.post(f"{internal_endpoint}/predict", headers=headers, json=payload)
print(f"\n=== Prediction Response (Status: {resp.status_code}) ===")
if resp.text:
    try:
        result = resp.json()
        print(json.dumps(result, indent=2))
    except Exception:
        print(resp.text)
else:
    print("Empty response")

print("\nEnd-to-end flow: Entity Key → Online Feature Store → Model Service → Prediction")

## 8. Clean Up

Drop the online service to stop the managed Postgres instance. Then run the teardown SQL from the QuickStart guide to remove all resources.

In [ ]:
# Clean up: delete feature views first, then drop the online service
print("Deleting registered feature views...")
for fv_name, fv_version in [("CUSTOMER_PROFILE_FEATURES", "V1"), ("CUSTOMER_TXN_AGG", "V1"), ("TXN_STREAM_VELOCITY", "V1")]:
    try:
        fs.delete_feature_view(
            fs.get_feature_view(fv_name, fv_version)
        )
        print(f"  Deleted: {fv_name}/{fv_version}")
    except Exception as e:
        print(f"  Skip {fv_name}/{fv_version}: {e}")

# Now drop the online service (stops the Postgres instance)
fs.drop_online_service()
print("\nOnline service dropped.")

# Drop the model service
try:
    mv.delete_service("FRAUD_SCORING_SVC")
    print("Model service FRAUD_SCORING_SVC dropped.")
except Exception as e:
    print(f"Model service cleanup: {e}")

print("\nTo complete teardown, run the SQL script from the QuickStart guide:")
print("  Projects > Workspaces > paste the teardown SQL > Run as ACCOUNTADMIN")

In [ ]:
-- ============================================================================
-- Teardown Script for Online Feature Store with Postgres: Fraud Detection Demo
-- ============================================================================
-- This script removes all resources created by the notebook.
-- Run this AFTER running fs.drop_online_service() in the cleanup cell above.
-- ============================================================================

USE ROLE ACCOUNTADMIN;

-- ============================================================================
-- SECTION 1: STOP SERVICES AND DROP COMPUTE POOL
-- ============================================================================

-- Stop all services in the compute pool and drop it
ALTER COMPUTE POOL IF EXISTS FS_DEMO_INFERENCE_POOL STOP ALL;
DROP COMPUTE POOL IF EXISTS FS_DEMO_INFERENCE_POOL;

-- ============================================================================
-- SECTION 2: DROP EXTERNAL ACCESS INTEGRATION
-- ============================================================================

DROP INTEGRATION IF EXISTS FRAUD_OFS_DEMO_ALLOW_ALL_INTEGRATION;

-- ============================================================================
-- SECTION 3: DROP DATABASE AND WAREHOUSE
-- ============================================================================

-- Drop database (also drops all schemas, tables, network rules, stages inside)
DROP DATABASE IF EXISTS FRAUD_OFS_DEMO_DB;

-- Drop warehouse
DROP WAREHOUSE IF EXISTS FS_DEMO_WH;

-- ============================================================================
-- SECTION 4: REVOKE ACCOUNT-LEVEL PERMISSIONS AND DROP ROLE
-- ============================================================================

REVOKE CREATE DATABASE ON ACCOUNT FROM ROLE FS_DEMO_ROLE;
REVOKE CREATE WAREHOUSE ON ACCOUNT FROM ROLE FS_DEMO_ROLE;
REVOKE CREATE COMPUTE POOL ON ACCOUNT FROM ROLE FS_DEMO_ROLE;
REVOKE BIND SERVICE ENDPOINT ON ACCOUNT FROM ROLE FS_DEMO_ROLE;
REVOKE IMPORT SHARE ON ACCOUNT FROM ROLE FS_DEMO_ROLE;
REVOKE EXECUTE TASK ON ACCOUNT FROM ROLE FS_DEMO_ROLE;
REVOKE EXECUTE MANAGED TASK ON ACCOUNT FROM ROLE FS_DEMO_ROLE;

DROP ROLE IF EXISTS FS_DEMO_ROLE;

-- ============================================================================
-- TEARDOWN COMPLETE
-- ============================================================================

SELECT 'Teardown complete! All demo resources have been removed.' AS STATUS;